# Training: Latent Domain Recovery

This notebook runs a full experiment from a JSON config file in `run-configs/`.  
Set `CONFIG_PATH` and `EXP_NAME` below to select the experiment you want to reproduce.

## 1. Setup

In [ ]:
import sys, os
# If running the notebook from notebooks/, add the repo root to the path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import json
import numpy as np
import tensorflow as tf
from core.train_utils import create_model, make_data_generator, train

print('TensorFlow:', tf.__version__)
print('GPUs available:', tf.config.list_physical_devices('GPU'))

## 2. Select experiment config

Available configs in `run-configs/`:

| File | Description | Experiment keys |
|---|---|---|
| `waveform-experiments.json` | 1-D waveform experiments (Gaussian & Legendre, natural & DST, 5 seeds each) | `GAUSSIAN-IDENTITY-EXP-{0..4}`, `GAUSSIAN-DST-EXP-{0..4}`, `LEGENDRE-IDENTITY-EXP-{0..4}`, `LEGENDRE-DST-EXP-{0..4}` |
| `mnist-experiments.json` | 2-D MNIST translation experiments | `MNIST-EXP-{0..2}` |
| `neuropixel-experiments.json` | Allen Brain Institute Neuropixel experiments (requires downloaded data) | `NEUROPIXEL-EXP-0` |

In [ ]:
CONFIG_PATH = os.path.join(REPO_ROOT, 'run-configs', 'waveform-experiments.json')
EXP_NAME    = 'GAUSSIAN-IDENTITY-EXP-0'  # key inside the config JSON

with open(CONFIG_PATH) as f:
    all_experiments = json.load(f)

exp_specs = all_experiments[EXP_NAME]

print(f'Experiment  : {EXP_NAME}')
print(f'Epochs      : {exp_specs["training_duration_in_epochs"]}')
print(f'Batch size  : {exp_specs["data_generator_params"]["batch_size"]}')
print(f'Input dim   : {exp_specs["data_generator_params"]["n_components"]}')

## 3. Generate training data

In [ ]:
seed = exp_specs.get('seed', 0)
np.random.seed(seed)
tf.random.set_seed(seed)

dg = make_data_generator(seed=seed, **exp_specs['data_generator_params'])
dg.reset_batch_counter()

batches = []
for b in range(exp_specs['num_training_batches']):
    batches.append(dg.sample_batch_of_data())
    if b % 10 == 0:
        print(f'Generated batch {b}/{exp_specs["num_training_batches"]}')

dataset_np = np.concatenate(batches, axis=0)
print(f'\nDataset shape: {dataset_np.shape}')

## 4. Build model

In [ ]:
model = create_model(**exp_specs['model_params'])
print('Model created.')

## 5. Train

Weights are saved every 10 epochs under `experiments/<EXP_NAME>/epochs/`.

In [ ]:
saving_dir = os.path.join(REPO_ROOT, 'experiments', EXP_NAME, 'epochs')
os.makedirs(saving_dir, exist_ok=True)

train(
    model=model,
    dataset_np=dataset_np,
    exp_specs=exp_specs,
    saving_dir=saving_dir,
    eager=False
)

print('Training complete.')
print(f'Weights saved to: {saving_dir}')

## 6. Save experiment specs alongside weights

In [ ]:
specs_path = os.path.join(REPO_ROOT, 'experiments', EXP_NAME, 'specs.json')
with open(specs_path, 'w') as f:
    def convert(obj):
        if isinstance(obj, np.ndarray): return obj.tolist()
        if isinstance(obj, np.float64): return float(obj)
        return obj
    json.dump(exp_specs, f, default=convert, indent=4)

print(f'Specs saved to: {specs_path}')

## 7. Quick peek at learned generators

For a full analysis open `evaluate.ipynb`.

In [ ]:
import matplotlib.pyplot as plt

G = model.lifting_layer.generators.numpy()   # (group_dims, D, D)

fig, axes = plt.subplots(1, G.shape[0], figsize=(5 * G.shape[0], 4))
if G.shape[0] == 1:
    axes = [axes]
for d, ax in enumerate(axes):
    im = ax.imshow(G[d], cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title(f'Generator dim {d}')
    plt.colorbar(im, ax=ax)
plt.suptitle('Learned group generators after training')
plt.tight_layout(); plt.show()